# Radiation shadow displacement

This notebook derives the horizontal displacement of cloud shadows between
1D and 3D MYSTIC direct irradiance fields.

Main steps:

1. Load corresponding 1D and 3D radiation datasets.
2. Average over the MYSTIC `run` dimension.
3. Estimate the displacement between 1D and 3D direct irradiance using
   periodic FFT cross-correlation.
4. Calculate displacement magnitude, direction, RMSE improvement, and
   effective shadow height.
5. Perform quick quality-control plots.
6. Store the derived quantities as a CSV file for later analysis.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

## Paths and case

In [ ]:
data_dir = Path(
    "/work/bb1555/user/kolja/datasets/c3sar_lindenberg"
)

case = "c3sar_exp024_DOM03_v01_20260606"

file_1d = data_dir / f"{case}_1D.nc"
file_3d = data_dir / f"{case}_3D.nc"

print(file_1d)
print(file_3d)

## Load datasets

In [ ]:
ds_1d = xr.load_dataset(file_1d).squeeze("exp", drop=True)
ds_3d = xr.load_dataset(file_3d).squeeze("exp", drop=True)

ds_1d

## Average MYSTIC runs

In [ ]:
edir_1d = ds_1d["edir"].mean(dim="run")
edir_3d = ds_3d["edir"].mean(dim="run")

print(edir_1d.dims)
print(edir_3d.dims)

## Shadow displacement

The 1D and 3D direct irradiance fields are compared using circular
FFT cross-correlation.

The domain is periodic, therefore the retrieved displacement is converted
from the FFT index into the corresponding signed periodic displacement.

The calculated shift is the shift that has to be applied to the 1D field
to align it with the 3D field.


## Shadow-displacement function

In [ ]:
def get_shadow_shift(
    edir_1d,
    edir_3d,
    ds,
    time_index
):
    """
    Estimate the horizontal displacement between 1D and 3D direct
    irradiance using circular FFT cross-correlation.

    Parameters
    ----------
    edir_1d : xarray.DataArray
        Run-mean 1D direct irradiance.
    edir_3d : xarray.DataArray
        Run-mean 3D direct irradiance.
    ds : xarray.Dataset
        Dataset containing time, SZA and SAA.
    time_index : int
        Time index to process.

    Returns
    -------
    dict
        Displacement and quality-control quantities for one timestamp.
    """

    # Surface irradiance
    e1 = edir_1d.isel(time=time_index).sel(zlev=0)
    e3 = edir_3d.isel(time=time_index).sel(zlev=0)

    a = e1.values
    b = e3.values

    # Metadata
    time = ds.time.isel(time=time_index).values
    sza = float(ds.sza.isel(time=time_index))
    saa = float(ds.saa.isel(time=time_index))

    # Check whether usable data exist
    if not np.isfinite(a).any() or not np.isfinite(b).any():

        return {
            "time_index": time_index,
            "time": time,
            "sza": sza,
            "saa": saa,
            "valid": False,
            "shift_x_px": np.nan,
            "shift_y_px": np.nan,
            "shift_x_m": np.nan,
            "shift_y_m": np.nan,
            "displacement_m": np.nan,
            "direction_deg": np.nan,
            "rmse_before": np.nan,
            "rmse_after": np.nan,
            "rmse_improvement": np.nan,
            "rmse_improvement_rel": np.nan,
            "h_eff_m": np.nan,
        }

    # Remove spatial mean
    a_anom = a - np.nanmean(a)
    b_anom = b - np.nanmean(b)

    # Circular FFT cross-correlation
    fa = np.fft.fft2(a_anom)
    fb = np.fft.fft2(b_anom)

    corr = np.fft.ifft2(
        fb * np.conj(fa)
    ).real

    iy, ix = np.unravel_index(
        np.nanargmax(corr),
        corr.shape
    )

    ny, nx = a.shape

    # Convert FFT indices to signed periodic shifts
    shift_y = iy if iy <= ny // 2 else iy - ny
    shift_x = ix if ix <= nx // 2 else ix - nx

    # Approximate grid spacing from coordinates
    lat_mean = float(e1.lat.mean())

    dx = (
        np.mean(np.diff(e1.lon.values))
        * 111_320
        * np.cos(np.deg2rad(lat_mean))
    )

    dy = (
        np.mean(np.diff(e1.lat.values))
        * 111_320
    )

    # Physical displacement components
    shift_x_m = shift_x * dx
    shift_y_m = shift_y * dy

    displacement_m = np.sqrt(
        shift_x_m**2 +
        shift_y_m**2
    )

    # Direction:
    # 0° = north
    # 90° = east
    # 180° = south
    # 270° = west
    direction_deg = (
        np.degrees(
            np.arctan2(
                shift_x_m,
                shift_y_m
            )
        ) % 360
    )

    # Apply shift to 1D field
    a_shifted = np.roll(
        a,
        shift=(shift_y, shift_x),
        axis=(0, 1)
    )

    # RMSE before and after shifting
    rmse_before = np.sqrt(
        np.nanmean((a - b)**2)
    )

    rmse_after = np.sqrt(
        np.nanmean((a_shifted - b)**2)
    )

    rmse_improvement = (
        rmse_before - rmse_after
    )

    if rmse_before > 0:
        rmse_improvement_rel = (
            rmse_improvement / rmse_before
        )
    else:
        rmse_improvement_rel = np.nan

    # Effective displacement height
    tan_sza = np.tan(
        np.deg2rad(sza)
    )

    if np.abs(tan_sza) > 1e-12:
        h_eff_m = (
            displacement_m / tan_sza
        )
    else:
        h_eff_m = np.nan

    return {
        "time_index": time_index,
        "time": time,
        "sza": sza,
        "saa": saa,
        "valid": True,
        "shift_x_px": shift_x,
        "shift_y_px": shift_y,
        "shift_x_m": shift_x_m,
        "shift_y_m": shift_y_m,
        "displacement_m": displacement_m,
        "direction_deg": direction_deg,
        "rmse_before": rmse_before,
        "rmse_after": rmse_after,
        "rmse_improvement": rmse_improvement,
        "rmse_improvement_rel": rmse_improvement_rel,
        "h_eff_m": h_eff_m,
    }

## Process all timestamps

In [ ]:
results = []

for ti in range(ds_1d.sizes["time"]):

    result = get_shadow_shift(
        edir_1d,
        edir_3d,
        ds_1d,
        time_index=ti
    )

    results.append(result)

df_shift = pd.DataFrame(results)

df_shift

## Define QC subset

In [ ]:
df_plot = df_shift[
    (df_shift["time_index"] >= 4)
    & (df_shift["valid"])
].copy()

print(
    f"Using {len(df_plot)} of "
    f"{len(df_shift)} timestamps for QC."
)

In [ ]:
# Temporary QC choice:
# The first four timestamps contain very little cloud/shadow structure,
# causing unstable FFT displacement estimates.
#
# This manual exclusion should later be replaced by an objective
# cloudiness / shadow-content criterion.

## Main QC plot

In [ ]:
fig, axes = plt.subplots(
    4,
    1,
    figsize=(10, 10),
    sharex=True,
    constrained_layout=True
)

# ------------------------------------------------------------------
# Shadow displacement
# ------------------------------------------------------------------

axes[0].plot(
    df_plot["time"],
    df_plot["displacement_m"],
    marker="o"
)

axes[0].set_ylabel(
    "Displacement [m]"
)

axes[0].set_title(
    "Retrieved cloud-shadow displacement"
)

axes[0].grid()


# ------------------------------------------------------------------
# Shift components
# ------------------------------------------------------------------

axes[1].plot(
    df_plot["time"],
    df_plot["shift_x_px"],
    marker="o",
    label="x shift"
)

axes[1].plot(
    df_plot["time"],
    df_plot["shift_y_px"],
    marker="o",
    label="y shift"
)

axes[1].set_ylabel(
    "Shift [px]"
)

axes[1].legend()
axes[1].grid()


# ------------------------------------------------------------------
# RMSE
# ------------------------------------------------------------------

axes[2].plot(
    df_plot["time"],
    df_plot["rmse_before"],
    marker="o",
    label="Before shift"
)

axes[2].plot(
    df_plot["time"],
    df_plot["rmse_after"],
    marker="o",
    label="After shift"
)

axes[2].set_ylabel(
    "RMSE [W m$^{-2}$]"
)

axes[2].legend()
axes[2].grid()


# ------------------------------------------------------------------
# Effective height
# ------------------------------------------------------------------

axes[3].plot(
    df_plot["time"],
    df_plot["h_eff_m"],
    marker="o",
    label=r"$h_{\mathrm{eff}} = d/\tan(\mathrm{SZA})$"
)

axes[3].set_ylabel(
    "Effective height [m]"
)

axes[3].set_xlabel(
    "Time"
)

axes[3].legend()
axes[3].grid()

plt.show()

## Solar-angle QC

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(11, 4.5),
    constrained_layout=True
)

# displacement vs tan(SZA)
axes[0].scatter(
    np.tan(
        np.deg2rad(
            df_plot["sza"]
        )
    ),
    df_plot["displacement_m"]
)

axes[0].set_xlabel(
    r"$\tan(\mathrm{SZA})$"
)

axes[0].set_ylabel(
    "Shadow displacement [m]"
)

axes[0].grid()


# direction vs SAA
axes[1].scatter(
    df_plot["saa"],
    df_plot["direction_deg"]
)

axes[1].set_xlabel(
    "Solar azimuth angle [°]"
)

axes[1].set_ylabel(
    "Displacement direction [°]"
)

axes[1].grid()

plt.show()

## summary statistics

In [ ]:
df_plot[
    [
        "displacement_m",
        "h_eff_m",
        "rmse_before",
        "rmse_after",
        "rmse_improvement_rel",
    ]
].describe()

## Export

The complete displacement table is stored, including timestamps that are
currently excluded from the QC analysis.

No cloud-based filtering is applied at this stage. This allows an objective
quality criterion to be introduced later without recalculating the radiation
displacement fields.


In [ ]:
output_dir = Path(
    "/work/bb1376/user/daniel/notebooks/cesar/cesar1own/02_3D-Rad-Effects"
) / "derived" / "shadow_displacement"

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

output_file = output_dir / f"{case}_shadow_displacement.csv"

df_shift.to_csv(
    output_file,
    index=False
)

print(f"Saved:\n{output_file}")

## Final sanity check

In [ ]:
df_check = pd.read_csv(
    output_file,
    parse_dates=["time"]
)

df_check.head()

In [ ]:
print(
    f"Rows: {len(df_check)}"
)

print(
    f"Valid timestamps: "
    f"{df_check['valid'].sum()}"
)